In [6]:
from typing import List, Dict
import ujson as json
from pathlib import Path
from openai import OpenAI
from datetime import datetime


# 1) Helper functions for Dataset Generation

## Generating slices from chunks

In [7]:
# turning chunks to slices

CHUNK_DIR = Path("data/rag_chunks_v2")

def load_chunks(path: Path) -> List[Dict]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)  # your file seems to be a JSON list



def make_slices_from_chunks(chunks: List[Dict],
                            max_chars: int = 5000,
                            min_chars: int = 2000,
                        ) -> List[Dict]:

    slices = []
    cur_text = []
    cur_len = 0
    cur_chunk_ids = []

    # assume chunks are already in reading order
    for ch in chunks:
        if ch.get("type") != "paragraph":
            continue
        t = (ch.get("content") or "").strip()
        if not t:
            continue

        if cur_len + len(t) > max_chars and cur_len >= min_chars:
            
            # close current slice
            slice_idx = len(slices)
            doc_id = ch["id"].split("_")[0]   # adjust if you store doc_id elsewhere
            
            slices.append({
                "doc_id": doc_id,
                "slice_id": f"{doc_id}_slice_{slice_idx}",
                "text": "\n\n".join(cur_text),
                "chunk_ids": cur_chunk_ids,
            })
            cur_text, cur_len, cur_chunk_ids = [], 0, []

        # add to current slice
        cur_text.append(t)
        cur_len += len(t)
        cur_chunk_ids.append(ch["id"])

    # flush last slice
    if cur_text:
        doc_id = chunks[0]["id"].split("_")[0]
        slice_idx = len(slices)
        
        slices.append({
            "doc_id": doc_id,
            "slice_id": f"{doc_id}_slice_{slice_idx}",
            "text": "\n\n".join(cur_text),
            "chunk_ids": cur_chunk_ids,
        })

    return slices

In [9]:
# testing slices

path = CHUNK_DIR / "1904.07640v1.rag.chunks.json"
chunks = load_chunks(path)
slices = make_slices_from_chunks(chunks)

print(len(slices), "slices")
print(slices[0]["text"][:2000]) 

14 slices
Post-market medical device surveillance is a challenge facing manufacturers, regulatory agencies, and health care providers. Electronic health records are valuable sources of real world evidence for assessing device safety and tracking device-related patient outcomes over time. However, distilling this evidence remains challenging, as information is fractured across clinical notes and structured records. Modern machine learning methods for machine reading promise to unlock increasingly complex information from text, but face barriers due to their reliance on large and expensive hand-labeled training sets. To address these challenges, we developed and validated state-of-the-art deep learning methods that identify patient outcomes from clinical notes without requiring hand-labeled training data. Using hip replacements -one of the most common implantable devices- as a test case, our methods accurately extracted implant details and reports of complications and pain from electroni

## LLM Call

In [10]:
# API Calls to ChatGPT 

from dotenv import load_dotenv

load_dotenv("keys.env") 
client = OpenAI()
MODEL = "gpt-5-nano"  


def get_response(prompt: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        input=[{"role": "user",
                "content": [{"type": "input_text", "text": prompt}]}],
    )
    return resp.output_text.strip()


In [11]:
# --- helper to safely extract JSON ---
def extract_json(raw: str) -> dict:
    raw = raw.strip()

    # Strip ``` fences if present
    if raw.startswith("```"):
        lines = raw.splitlines()
        lines = [ln for ln in lines if not ln.strip().startswith("```")]
        raw = "\n".join(lines).strip()

    # First try: as-is
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Second try: escape backslashes (for \alpha, \nabla, etc.)
    try:
        fixed = raw.replace("\\", "\\\\")
        return json.loads(fixed)
        
    except json.JSONDecodeError as e:
        print("JSON parse error even after fixing backslashes:", e)
        print("Raw output:\n", raw)
        # Fallback: return empty structure so rest of pipeline doesn't crash
        return {"qas": []}

# 2) Dataset Generation with Self-Instruct

In [12]:
SELF_INSTRUCT_PROMPT = """
You are a helpful assistant reading a research paper excerpt.

TEXT:
\"\"\"{text}\"\"\"

Generate {n_qas} diverse question-answer pairs that can be answered *directly and unambiguously* from this text alone.

Requirements:
- Cover different types: definitions, methods, motivations, comparisons, results.
- Make questions specific, not vague.
- Answers should be concise and copy or paraphrase the text.
- Return JSON as a list under key "qas", like:
  {{"qas": [{{"question": "...", "answer": "..."}}, ...]}}
"""

In [13]:
QA = Dict[str, str]

def generate_qas_self_instruct(doc_id: str, text: str, n_qas: int = 5) -> List[QA]:
    
    prompt = SELF_INSTRUCT_PROMPT.format(text=text, n_qas=n_qas)
    response = get_response(prompt)
    data = extract_json(response)

    qas: List[QA] = []
    for qa in data.get("qas", []):
        qas.append({
            "question": qa["question"],
            "answer": qa["answer"],
            "doc_id": doc_id,
            "source_text": text,
        })
    return qas

### sample QA generation

In [14]:
slice0 = slices[0]
qas = generate_qas_self_instruct(
    doc_id=slice0["doc_id"],
    text=slice0["text"],
    n_qas=5,
)

for qa in qas:
    print("Q:", qa["question"])
    print("A:", qa["answer"])
    print("----")

Q: What challenge makes distilling evidence from electronic health records difficult in post-market device surveillance?
A: Information is fractured across clinical notes and structured records.
----
Q: What type of methods did the researchers develop to identify patient outcomes without hand-labeled training data?
A: State-of-the-art deep learning methods that identify patient outcomes from clinical notes without requiring hand-labeled training data.
----
Q: In the hip replacement test case, what were the reported precision, recall, and F1 scores for extracting implant details and complications from EHRs?
A: Precision: 96.3%; Recall: 98.5%; F1: 97.4%.
----
Q: How did the authors' method perform relative to rule-based methods?
A: It improved classification performance by about 12.75–13.0% over rule-based methods.
----
Q: What do the authors claim their methods provide for national medical device surveillance?
A: A scalable solution requiring orders of magnitude less hand-labeled traini

### Building and saving dataset from all chunks

In [16]:
# create folder: data/QnA/
save_dir = Path("data") / "QnA"
save_dir.mkdir(parents=True, exist_ok=True)

OUT_PATH = save_dir / "eval_qas_self_instruct.jsonl"
DATASET_NAME = "arxiv_scale_rag_self_instruct_v1" 

In [17]:


def build_eval_qas_jsonl(
    chunk_dir: Path = CHUNK_DIR,
    out_path: Path = OUT_PATH,
    n_qas_per_slice: int = 5,
    max_docs: int | None = None,   
):
    with out_path.open("w", encoding="utf-8") as out_f:
        for i, path in enumerate(sorted(chunk_dir.glob("*.rag.chunks.json"))):
            if max_docs is not None and i >= max_docs:
                break

            chunks = load_chunks(path)
            slices = make_slices_from_chunks(chunks, max_chars=3000, min_chars=500)

            for slice_idx, s in enumerate(slices):
                qas = generate_qas_self_instruct(
                    doc_id=s["doc_id"],
                    text=s["text"],
                    n_qas=n_qas_per_slice,
                )

                for qa_idx, qa in enumerate(qas):
                    record = {
                        # identifiers
                        "id": f"{s['doc_id']}_slice{slice_idx}_qa{qa_idx}",
                        "dataset": DATASET_NAME,
                        "doc_id": s["doc_id"],
                        "slice_id": s["slice_id"],
                        "slice_index": slice_idx,
                        "chunk_ids": s["chunk_ids"],

                        # QA
                        "question": qa["question"],
                        "answer": qa["answer"],

                        # optional, but useful for debugging / later analysis
                        "source_text": s["text"],  

                        # meta
                        "generator_model": MODEL,  
                        "created_at": datetime.utcnow().isoformat(),
                    }
                    out_f.write(json.dumps(record) + "\n")

            print(f"Processed {path.name}: {len(slices)} slices")


In [19]:
build_eval_qas_jsonl(max_docs=10)  # try on 3 papers first

Processed 1904.07640v1.rag.chunks.json: 24 slices
Processed 1904.07687v4.rag.chunks.json: 11 slices
Processed 1904.07698v2.rag.chunks.json: 24 slices
Processed 1904.07916v1.rag.chunks.json: 19 slices
Processed 1904.07964v1.rag.chunks.json: 12 slices
Processed 1904.07969v1.rag.chunks.json: 8 slices
Processed 1904.07998v2.rag.chunks.json: 14 slices
JSON parse error even after fixing backslashes: Unexpected character in found when decoding object value
Raw output:
 {"qas": [{"question": "What is the meta-grammar M and what does it do in the model?", "answer": "M is a probabilistic context free grammar (PCFG) for generating L; at each step, a PCFG rewrite rule is chosen uniformly at random from the applicable set to define the prior on L-systems, P(L)."}, {"question": "How does the PCFG generate the F-rule?", "answer": "It begins at 'Start' and applies production rules until the string consists of only terminal symbols { F, G, +, -, ' ' }.}, {"question": "What constraints are placed on L-s

In [20]:
save_log(run_log, Path("run_log.graph_rag_v1.jsonl"))

NameError: name 'save_log' is not defined